# Vigenere - classical baseline

Results of the learning-free solver (Kasiski vote + coset coincidence search +
reduction to the fundamental period + chi-square shift recovery) on the same
grid used for the neural model.

Two conditions are reported so period recovery and shift recovery can be read
apart: **pipeline** (period estimated from the ciphertext) and **oracle** (true
period supplied). No checkpoint is needed.

In [ ]:
import os, sys
from pathlib import Path

def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if in_colab():
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = Path('/content/drive/MyDrive/cryptanalysis-ai')
    %pip install -q pytorch-lightning torchmetrics matplotlib
else:
    REPO_DIR = Path.cwd()
    if REPO_DIR.name == 'notebooks':
        REPO_DIR = REPO_DIR.parent

os.environ['CRYPT_DATA_DIR'] = str(REPO_DIR / 'data')
os.environ['CRYPT_CKPT_DIR'] = str(REPO_DIR / 'checkpoints')
os.environ['CRYPT_LOG_DIR']  = str(REPO_DIR / 'tb_logs')
sys.path.insert(0, str(REPO_DIR / 'src'))
print('repo:', REPO_DIR)


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import config
from models.vigenere import classical as vcl, inference as vinf, evaluation as ev

clean = ev.load_clean_text(config.data_file("validation_set.txt"), config.ALPHABET)
print(f"validation: {len(clean):,} characters")


## Decoding grid

`searched_max_k` shows the largest period actually searched: on short texts a
candidate is skipped when it would leave fewer than two characters per coset.

In [ ]:
TEST_KEY_LENS = [3, 5, 8, 12, 20, 32]
TEST_LENGTHS  = [128, 256, 512]
N = 500

classical = ev.run_classical_eval(clean, TEST_LENGTHS, TEST_KEY_LENS, config.ALPHABET,
                                  num_samples=N, min_k=vcl.MIN_KEY_LEN, max_k=vcl.MAX_KEY_LEN)

ev.print_grid(classical, "pipeline (period estimated)", fields=("char_acc", "key_acc", "avg_ms"))
print()
ev.print_grid(classical, "oracle (true period supplied)",
              fields=("oracle_char_acc", "exact_key", "key_sym_acc"))

cells = [classical[k][L] for k in TEST_KEY_LENS for L in TEST_LENGTHS]
print(f"\nmean pipeline CHAR% : {np.mean([c['char_acc'] for c in cells]):.1f}")
print(f"mean period acc %   : {np.mean([c['key_acc'] for c in cells]):.1f}")
print(f"mean oracle CHAR%   : {np.mean([c['oracle_char_acc'] for c in cells]):.1f}")


## Heatmaps

Left: pipeline character accuracy. Right: oracle character accuracy. The gap
between them is the cost of estimating the period.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, field, title in zip(axes, ["char_acc", "oracle_char_acc"],
                            ["pipeline CHAR%", "oracle CHAR%"]):
    grid = np.array([[classical[k][L][field] for L in TEST_LENGTHS] for k in TEST_KEY_LENS])
    im = ax.imshow(grid, vmin=0, vmax=100, cmap="viridis", aspect="auto")
    ax.set_xticks(range(len(TEST_LENGTHS)), TEST_LENGTHS)
    ax.set_yticks(range(len(TEST_KEY_LENS)), TEST_KEY_LENS)
    ax.set_xlabel("text length"); ax.set_ylabel("key length"); ax.set_title(title)
    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            ax.text(j, i, f"{grid[i, j]:.0f}", ha="center", va="center",
                    color="white" if grid[i, j] < 60 else "black")
fig.colorbar(im, ax=axes, shrink=0.85)
plt.savefig(config.STORAGE_ROOT / "classical_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


## Effect of the fundamental-period reduction

The coset coincidence rate is the same for every multiple of the true period,
so the raw minimiser can return a multiple. This compares the estimator with
and without the reduction step (`tol=None`) on period accuracy.

In [ ]:
import random
char2idx, idx2char = vinf.build_vocab(config.ALPHABET)
ABL_N = 200
print(f"{'k':>3} {'L':>5} | {'reduced':>8} {'raw':>8}")
for k in TEST_KEY_LENS:
    for L in TEST_LENGTHS:
        ok_r = ok_raw = 0
        for i in range(ABL_N):
            s = (i * L * 13) % max(1, len(clean) - L)
            key = ev.get_fixed_key(k, i, config.ALPHABET)
            cipher, _ = vinf.encrypt_vigenere(clean[s:s+L], key, char2idx, idx2char)
            ok_r   += vcl.estimate_key_length_combined(cipher) == k
            ok_raw += vcl.estimate_key_length_combined(cipher, tol=None) == k
        print(f"{k:>3} {L:>5} | {ok_r/ABL_N*100:>8.1f} {ok_raw/ABL_N*100:>8.1f}")


## PATE vs classical

Both solvers under the same two conditions on the grid above. The pipeline
columns are the fair comparison, since each solver estimates the period itself;
the oracle columns supply the true period to both and isolate shift recovery.

Requires the trained PATE checkpoint in `checkpoints/vigenere_pate`.

In [ ]:
import torch
from models.vigenere.model_pate import VigenereSolver

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# saved as "nocycle-best-hard-...ckpt"; fall back to the older name
ckpt_dir = config.CHECKPOINT_DIR / "vigenere_pate"
cands = sorted(ckpt_dir.glob("*best-hard*.ckpt")) or sorted(ckpt_dir.glob("*best-acc*.ckpt"))
ckpt = cands[-1]
model = VigenereSolver.load_from_checkpoint(str(ckpt), map_location=DEVICE)
model.to(DEVICE).eval(); model.freeze()
print("checkpoint:", ckpt.name)

neural = ev.run_neural_eval(model, clean, TEST_LENGTHS, TEST_KEY_LENS,
                            config.ALPHABET, num_samples=N, device=DEVICE)
ev.print_comparison(neural, classical)
